In [2]:
import torch
from torch import nn
import numpy as np
import pandas as pd
import torch
import torch.optim as optim

In [3]:
df = pd.read_csv("../../Data/Final/Imputation/changi_imp_final.csv")
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '../../Data/Final/Imputation/changi_imp_final.csv'

In [ ]:
## THIS CODE IS TO REFORMAT THE TIME INDEX IN THE DATASET ##

# Function to create a time index
def create_time_index(df):
    bimonthly_map = {"Jan-Feb": 1, "Mar-Apr": 2, "May-Jun": 3, "Jul-Aug": 4, "Sep-Oct": 5, "Nov-Dec": 6}
    
    df["Year"] = df["Date"].str[-4:].astype(int)  # Extract year
    df["Period"] = df["Date"].str[:-5].map(bimonthly_map)  # Extract period and map it

    df["time_index"] = (df["Year"] - 2000) * 6 + df["Period"]  # Compute time index
    df = df.drop(columns=["Year", "Period"])  # Drop extra columns
    
    return df

df = create_time_index(df)
df.to_csv("changi_test_long_time_index.csv", index=False)


            x         y          Date      Value  time_index
0  103.964566  1.350459  Mar-Apr 2000  23.775064           2
1  103.964566  1.351269  Mar-Apr 2000  23.403744           2
2  103.964566  1.352079  Mar-Apr 2000  22.966872           2
3  103.964566  1.352889  Mar-Apr 2000  22.417757           2
4  103.965377  1.348838  Mar-Apr 2000  23.699749           2


In [ ]:
## THIS CODE IS TO CREATE ROLLING WINDOW SEQUENCES FOR A WINDOW SIZE OF 9 ##

# Define sequence length
sequence_length = 9 * 6  # 54 time steps (input)
target_length = 12  # Predict the next 12 time steps

# Group by (x, y) and process each time series separately
grouped = df.groupby(["x", "y"])

# Lists to store input sequences and target sequences separately
input_sequences = []
target_sequences = []
locations = []

# Iterate over each coordinate group
for (x, y), group in grouped:
    # Sort by time index
    group = group.sort_values(by="time_index")

    # Extract LST values
    values = group["Value"].values

    # Generate sequences
    for i in range(len(values) - sequence_length - target_length + 1):
        input_seq = values[i : i + sequence_length]  # Past 54 values
        target_seq = values[i + sequence_length : i + sequence_length + target_length]  # Next 12 values
        
        # Store sequences separately
        input_sequences.append([x, y] + list(input_seq))
        target_sequences.append([x, y] + list(target_seq))
        locations.append([x, y])

# Define column names
input_columns = ["x", "y"] + [f"LST_t-{i}" for i in range(sequence_length, 0, -1)]
target_columns = ["x", "y"] + [f"Target_t+{i}" for i in range(1, target_length + 1)]

# Convert to DataFrames
input_df = pd.DataFrame(input_sequences, columns=input_columns)
target_df = pd.DataFrame(target_sequences, columns=target_columns)

'''
# Save input and target sequences as separate CSV files
input_df.to_csv("changi_inputs.csv", index=False)
target_df.to_csv("changi_targets.csv", index=False)
'''

# Print sample sequences
print("Input sequences sample:")
print(input_df.head())
print("\nTarget sequences sample:")
print(target_df.head())


            x         y   LST_t-54   LST_t-53   LST_t-52   LST_t-51  \
0  103.964566  1.350459  23.775064  27.537897  27.519318  32.314507   
1  103.964566  1.350459  27.537897  27.519318  32.314507  23.391225   
2  103.964566  1.350459  27.519318  32.314507  23.391225  29.400195   
3  103.964566  1.350459  32.314507  23.391225  29.400195  29.712163   
4  103.964566  1.350459  23.391225  29.400195  29.712163  27.845558   

    LST_t-50   LST_t-49   LST_t-48   LST_t-47  ...    LST_t-9    LST_t-8  \
0  23.391225  29.400195  29.712163  27.845558  ...  25.289893  26.789666   
1  29.400195  29.712163  27.845558  24.015326  ...  26.789666  27.538707   
2  29.712163  27.845558  24.015326  25.797058  ...  27.538707  21.919449   
3  27.845558  24.015326  25.797058  29.891817  ...  21.919449  28.271844   
4  24.015326  25.797058  29.891817  31.118223  ...  28.271844  23.193389   

     LST_t-7    LST_t-6    LST_t-5    LST_t-4    LST_t-3    LST_t-2  \
0  27.538707  21.919449  28.271844  23.193389

In [8]:
# Define sequence length
sequence_length = 9 * 6  # 54 time steps (input)
target_length = 12  # Predict the next 12 time steps

# Group by (x, y) and process each time series separately
grouped = df.groupby(["x", "y"])

# List to store sequences
all_sequences = []

# Iterate over each coordinate group
for (x, y), group in grouped:
    # Sort by time index
    group = group.sort_values(by="time_index")

    # Extract LST values
    values = group["Value"].values

    # Generate sequences
    for i in range(len(values) - sequence_length - target_length + 1):
        input_seq = values[i : i + sequence_length]  # Past 54 values
        target_seq = values[i + sequence_length : i + sequence_length + target_length]  # Next 12 values
        all_sequences.append([x, y] + list(input_seq) + list(target_seq))

# Define column names
columns = ["x", "y"] + [f"LST_t-{i}" for i in range(sequence_length, 0, -1)] + [f"Target_t+{i}" for i in range(1, target_length + 1)]

# Convert to DataFrame
sequences_df = pd.DataFrame(all_sequences, columns=columns)

# Save as CSV
sequences_df.to_csv("changi_sequences_multistep.csv", index=False)

# Print sample sequences
print(sequences_df.head())

            x         y   LST_t-54   LST_t-53   LST_t-52   LST_t-51  \
0  103.964566  1.350459  23.775064  27.537897  27.519318  32.314507   
1  103.964566  1.350459  27.537897  27.519318  32.314507  23.391225   
2  103.964566  1.350459  27.519318  32.314507  23.391225  29.400195   
3  103.964566  1.350459  32.314507  23.391225  29.400195  29.712163   
4  103.964566  1.350459  23.391225  29.400195  29.712163  27.845558   

    LST_t-50   LST_t-49   LST_t-48   LST_t-47  ...  Target_t+3  Target_t+4  \
0  23.391225  29.400195  29.712163  27.845558  ...   23.694784   26.360794   
1  29.400195  29.712163  27.845558  24.015326  ...   26.360794   26.155256   
2  29.712163  27.845558  24.015326  25.797058  ...   26.155256   26.584512   
3  27.845558  24.015326  25.797058  29.891817  ...   26.584512   26.959252   
4  24.015326  25.797058  29.891817  31.118223  ...   26.959252   28.922135   

   Target_t+5  Target_t+6  Target_t+7  Target_t+8  Target_t+9  Target_t+10  \
0   26.155256   26.584512 